# E5 step 1 · Hash check: the validation leak and the benchmark's copies of training images

Task **E5** in `Essential_task_delegation/Next_Steps_Roadmap.md` (built 26 Oct 2026). **CPU only:** set the accelerator to None, so it uses no GPU hours.

**Why**
- **The validation leak:**
  - Ridita's audit found that 40 of our 645 validation pairs (155,812 of 1,808,111 destroyed pixels) are near-copies of training pairs: xBD's `socal_fire` and `woolsey_fire` share before-images.
  - A check on your PC, hashing xBD's original files, found 21 such pairs (80,709 pixels).
  - This run settles the number on DisasterM3's own copies of the images, as the audit did.
- **The benchmark:** our 7B and R11 were trained on DisasterM3 Instruct images. If a benchmark test image copies an Instruct image, their benchmark scores (O2, L0, H1 V2, H2) are too high on the questions that use it.

**What it does**
1. **The split:** rebuilds the 6,443 pairs and the 645-image validation split exactly as every D4 run did (R11's cells, verbatim), then checks it: 645 images, 560 of them from xBD, 1,808,111 destroyed pixels.
2. **Fingerprints:** for every image file in DisasterM3 Instruct and in the benchmark, an MD5 of the file, an MD5 of the decoded pixels, a 64-bit difference hash (dHash) and a 32 × 32 grey thumbnail.
3. **The validation leak:**
   - a validation pair leaks if its before-image has the same pixels as a training pair's before-image (the audit's rule), or its after-image has the same pixels as a training after-image;
   - near-copies (dHash within 6 of 64 bits and thumbnails correlating at 0.95 or more) are listed separately and not counted.
4. **The benchmark's copies:**
   - every benchmark image is compared with every Instruct image, for exact and near-copies;
   - then each question of the six scored tasks is flagged if its after-image (or only its before-image) is a copy, using the question IDs that H1 V2, L0 and H2 use.
5. **Pictures:** near-copies side by side, for a check by eye.

**Expectation, fixed before the run (26 Oct)**
- **Validation:** 21 to 40 leaked pairs. All 21 from the PC check must be among them, since they came from identical xBD files.
- **The pool:** about 264 before-images shared by two pairs (the audit's count).
- **The benchmark:** no number was predicted. Copies, if any, are expected mostly from events stored under two names. If more than 1% of the six tasks' optical questions have a copied after-image, O2's numbers get a clean re-score on your PC.

**How to run on Kaggle** (account 2)
1. Session options: Accelerator **None**.
2. Inputs: `disasterm3-mirror` (abrarmohammedtanzim) and `disasterm3-bench-mirror-v2` (redwannewaz).
3. Smoke test, then the full run:
   - Run All with `SMOKE_TEST = True` (a few minutes). It fingerprints a small sample that includes one known twin pair, which must show up as a leak.
   - Then set `SMOKE_TEST = False` and use Save & Run All (Commit), about 30–60 minutes.
4. Outputs in `/kaggle/working/E5-step1/`:
   - `README.md` and `summary.json`;
   - `val_train_leaks_hash.csv` and `pool_twin_groups.csv`;
   - `bench_copies.csv` and `bench_questions_flagged.csv`;
   - `fingerprints_instruct.csv` and `fingerprints_bench.csv`;
   - `review/`, the pictures.

In [1]:
# ── Cell 1: Configuration ─────────────────────────────────────────────────
import csv
import glob
import hashlib
import json
import os
import re
import shutil
import time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2
import numpy as np

SMOKE_TEST = False                     # True = fingerprint a small sample only (checks the code, not the answer)
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = INPUT_ROOT / "datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct"
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/E5-step1")
MASK_TMP = "/tmp/combined_masks"       # where R11's cells write the 4-class masks

NEAR_DHASH, NEAR_CORR = 6, 0.95        # a near-copy: dHash within 6 of 64 bits AND grey thumbnails correlating >= 0.95
REVIEW_DHASH, REVIEW_CORR = 10, 0.85   # looser: pictured for review only, never counted as a copy
DARK, FLAT = 0.5, 4.0                  # never near-matched: images over half black, or thumbnails with grey std < 4
WORKERS = os.cpu_count() or 4

EXPECTED = {"pairs": 6443, "val": 645, "val_xbd": 560, "val_destroyed_px": 1808111}
AUDIT = {"leaked_pairs": 40, "leaked_destroyed_px": 155812, "shared_before_images_in_pool": 264}
# The PC check of 26 Oct (xBD's original files): its 21 leaked validation pairs, by after-image stem
PC_21 = ["socal_fire_00000081", "socal_fire_00000220", "socal_fire_00000222", "socal_fire_00000301",
         "socal_fire_00000341", "socal_fire_00000348", "socal_fire_00000436", "socal_fire_00000455",
         "socal_fire_00000497", "woolsey_fire_00000009", "woolsey_fire_00000030", "woolsey_fire_00000032",
         "woolsey_fire_00000474", "woolsey_fire_00000529", "woolsey_fire_00000573", "woolsey_fire_00000623",
         "woolsey_fire_00000632", "woolsey_fire_00000634", "woolsey_fire_00000703", "woolsey_fire_00000783",
         "woolsey_fire_00000810"]
KNOWN_TWIN = ("socal_fire_00000081_post_disaster.png", "woolsey_fire_00000017_post_disaster.png")   # validation, training

assert MANIFEST_PATH.exists(), f"No manifest at {MANIFEST_PATH}: attach abrarmohammedtanzim/disasterm3-mirror"
_bench = [Path(p).parent for p in sorted(glob.glob(str(INPUT_ROOT / "**" / "benchmark_release.json"), recursive=True))
          if (Path(p).parent / "test_images").is_dir()]
if not _bench:
    raise FileNotFoundError("Attach redwannewaz/disasterm3-bench-mirror-v2 (benchmark_release.json next to test_images/)")
BENCH_ROOT = _bench[0]
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "review").mkdir(exist_ok=True)
print(f"✓ E5 step 1{'  [SMOKE TEST]' if SMOKE_TEST else ''}: Instruct {DATA_ROOT}; benchmark {BENCH_ROOT}; "
      f"{WORKERS} worker(s); outputs {OUT_DIR}")

✓ E5 step 1: Instruct /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct; benchmark /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2; 4 worker(s); outputs /kaggle/working/E5-step1


## 1. The split: the same 645 validation images as every D4 run
Cell 2 is R11's (via R16c step 3), verbatim except that its mask folder is `MASK_TMP`. Cell 3 is R11's split, plus the destroyed-pixel check. So the split matches R1–R16c exactly.

In [2]:
# ── Cell 2: Load manifest and build the 4-class masks (R11's Cell 4, verbatim but for MASK_TMP) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")   # R11: the before-image

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs(MASK_TMP, exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"{MASK_TMP}/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
        "pre_image_path": resolve_path(mask_dict.get("pre_image_path")),   # R11: None if not found
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")
print(f"  Before-image not found: {sum(p['pre_image_path'] is None for p in pairs):,} (must be 0 for R11)")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0
  Before-image not found: 0 (must be 0 for R11)


In [3]:
# ── Cell 3: The validation split (identical to every D4 run) and its destroyed pixels ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def destroyed_px(pair):
    m = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
    return int((m == 3).sum()) if m is not None else 0

assert all(p["pre_image_path"] for p in pairs), "some pairs have no before-image"
POOL = []      # one row per pair: its split, file names, event and (validation only) destroyed pixels
for split, group in (("train", train_pairs), ("val", val_pairs)):
    for p in group:
        event, source = event_and_source(p["image_path"])
        POOL.append({"split": split, "post": name_of(p["image_path"]), "pre": name_of(p["pre_image_path"]),
                     "event": event, "source": source, "destroyed_px": destroyed_px(p) if split == "val" else None})
VAL = [r for r in POOL if r["split"] == "val"]
TRAIN = [r for r in POOL if r["split"] == "train"]
n_xbd = sum(r["source"] == "xBD" for r in VAL)
val_destroyed = sum(r["destroyed_px"] for r in VAL)
print(f"✓ Pairs {len(pairs):,}: training {len(TRAIN):,}, validation {len(VAL)} ({n_xbd} xBD); "
      f"destroyed pixels in validation {val_destroyed:,}")
assert (len(pairs), len(VAL), n_xbd) == (EXPECTED["pairs"], EXPECTED["val"], EXPECTED["val_xbd"]), "not the D4 split"
assert val_destroyed == EXPECTED["val_destroyed_px"], "the validation masks differ from the ones every D4 run scored"
val_posts = {r["post"] for r in VAL}
assert all(f"{s}_post_disaster.png" in val_posts for s in PC_21), "the PC check's 21 pairs aren't all in validation"
assert KNOWN_TWIN[0] in val_posts and KNOWN_TWIN[1] in {r["post"] for r in TRAIN}, "the known twin isn't split as expected"
print("✓ The split matches every D4 run; the PC check's 21 pairs are all validation pairs")

✓ Pairs 6,443: training 5,798, validation 645 (560 xBD); destroyed pixels in validation 1,808,111
✓ The split matches every D4 run; the PC check's 21 pairs are all validation pairs


## 2. Fingerprints of every image file
- **File MD5** finds byte-for-byte copies.
- **Pixel MD5** finds copies that were saved again.
- **dHash and the thumbnail** find near-copies: re-saved, recompressed, or slightly shifted or cropped.

Self-checks on made-up images run first.

In [4]:
# ── Cell 4: Fingerprints (file MD5, pixel MD5, dHash, darkness, 32x32 grey thumbnail) ──
import multiprocessing as mp

IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def fingerprint(path):
    """One image file -> its fingerprints. Never raises: an unreadable file gets an empty pixel hash."""
    cv2.setNumThreads(1)
    raw = Path(path).read_bytes()
    out = {"path": str(path), "file_md5": hashlib.md5(raw).hexdigest(), "pixel_md5": "", "dhash": 0,
           "dark": 1.0, "h": 0, "w": 0, "thumb": np.zeros((32, 32), np.uint8)}
    img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR) if raw else None
    if img is None:
        return out
    h, w = img.shape[:2]
    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(grey, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
    bits = (small[:, 1:] > small[:, :-1]).ravel()
    out.update(pixel_md5=hashlib.md5(f"{h}x{w}:".encode() + img.tobytes()).hexdigest(),
               dhash=int.from_bytes(np.packbits(bits).tobytes(), "big"),
               dark=round(float((grey < 8).mean()), 4), h=h, w=w,
               thumb=cv2.resize(grey, (32, 32), interpolation=cv2.INTER_AREA))
    return out

POPCOUNT8 = np.array([bin(i).count("1") for i in range(256)], np.uint8)

def bit_distance(a, b):
    """Hamming distance between uint64 values or arrays (broadcast)."""
    x = np.bitwise_xor(np.asarray(a, np.uint64), np.asarray(b, np.uint64))
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return POPCOUNT8[np.ascontiguousarray(x).view(np.uint8).reshape(*x.shape, 8)].sum(-1).astype(np.int16)

def thumb_corr(a, b):
    """Pearson correlation of paired rows of a and b (flattened thumbnails)."""
    a = a - a.mean(1, keepdims=True)
    b = b - b.mean(1, keepdims=True)
    den = np.sqrt((a * a).sum(1) * (b * b).sum(1))
    return np.where(den > 0, (a * b).sum(1) / np.maximum(den, 1e-9), 0.0)

def tier(d, c):
    """A pair's class: a near-copy is counted; a review match is only pictured."""
    if d <= NEAR_DHASH and c >= NEAR_CORR:
        return "near copy"
    return "review" if c >= REVIEW_CORR else None

# Self-checks on made-up images: the same pixels saved again, a recompressed copy, a 3-pixel shift, an unrelated image
def _texture(seed, size=512):
    rng = np.random.default_rng(seed)
    t = cv2.GaussianBlur(rng.integers(0, 256, (size, size, 3)).astype(np.float32), (0, 0), 6)
    return cv2.normalize(t, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

_chk = OUT_DIR / "_selfcheck"
_chk.mkdir(exist_ok=True)
_a = _texture(1)
cv2.imwrite(str(_chk / "a.png"), _a, [cv2.IMWRITE_PNG_COMPRESSION, 1])
cv2.imwrite(str(_chk / "a_again.png"), _a, [cv2.IMWRITE_PNG_COMPRESSION, 9])     # same pixels, other bytes
cv2.imwrite(str(_chk / "a_jpeg.jpg"), _a, [cv2.IMWRITE_JPEG_QUALITY, 85])          # recompressed
cv2.imwrite(str(_chk / "a_shift.png"), np.roll(_a, 3, axis=1))                     # shifted by 3 px
cv2.imwrite(str(_chk / "b.png"), _texture(2))                                      # unrelated
_f = {p.stem: fingerprint(p) for p in sorted(_chk.iterdir())}
shutil.rmtree(_chk)
_d = lambda x, y: int(bit_distance(_f[x]["dhash"], _f[y]["dhash"]))
_c = lambda x, y: float(thumb_corr(_f[x]["thumb"].reshape(1, -1).astype(np.float32),
                                   _f[y]["thumb"].reshape(1, -1).astype(np.float32))[0])
assert _f["a"]["pixel_md5"] == _f["a_again"]["pixel_md5"] and _f["a"]["file_md5"] != _f["a_again"]["file_md5"]
assert tier(_d("a", "a_jpeg"), _c("a", "a_jpeg")) == "near copy", (_d("a", "a_jpeg"), _c("a", "a_jpeg"))
assert tier(_d("a", "a_shift"), _c("a", "a_shift")) == "near copy", (_d("a", "a_shift"), _c("a", "a_shift"))
assert tier(_d("a", "b"), _c("a", "b")) is None and _d("a", "b") > REVIEW_DHASH, (_d("a", "b"), _c("a", "b"))
print(f"✓ Fingerprint self-checks passed (dHash: recompressed {_d('a', 'a_jpeg')}, shifted {_d('a', 'a_shift')}, "
      f"unrelated {_d('a', 'b')} of 64 bits)")

def list_images(root):
    """Every image file under root, one per file name (the first one found)."""
    seen, files, dup = set(), [], 0
    for p in sorted(Path(root).rglob("*")):
        if p.suffix.lower() in IMG_EXT:
            if p.name in seen:
                dup += 1
                continue
            seen.add(p.name)
            files.append(str(p))
    return files, dup

ins_files, ins_dup = list_images(DATA_ROOT / "train_images")
ben_files, ben_dup = list_images(BENCH_ROOT / "test_images")
print(f"Image files: Instruct {len(ins_files):,} (same name twice: {ins_dup}), "
      f"benchmark {len(ben_files):,} (same name twice: {ben_dup})")
if SMOKE_TEST:
    by_name = {Path(p).name: p for p in ins_files}
    twin_files = {by_name[n] for r in POOL if r["post"] in KNOWN_TWIN for n in (r["pre"], r["post"]) if n in by_name}
    ins_files = sorted(set(ins_files[:300]) | twin_files)
    ben_files = ben_files[:150]

def fingerprint_all(files, label):
    out, t0 = [], time.time()
    # Workers started by "fork" can call fingerprint(), which lives in this notebook; Python 3.14's default
    # start method on Linux ("forkserver") can't, so fork is asked for by name.
    ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None
    ex = ProcessPoolExecutor(max_workers=WORKERS, mp_context=ctx) if WORKERS > 1 else None
    results = ex.map(fingerprint, files, chunksize=16) if ex else map(fingerprint, files)
    for i, fp in enumerate(results, 1):
        out.append(fp)
        if i % 2000 == 0 or i == len(files):
            rate = i / max(time.time() - t0, 1e-6)
            print(f"  {label}: {i:,}/{len(files):,} ({rate:.0f} files/s; {(len(files) - i) / rate / 60:.1f} min left)")
    if ex:
        ex.shutdown()
    return out

def as_table(fps):
    T = {"names": [Path(f["path"]).name for f in fps], "paths": [f["path"] for f in fps],
         "pixel": [f["pixel_md5"] for f in fps], "file": [f["file_md5"] for f in fps],
         "dhash": np.array([f["dhash"] for f in fps], np.uint64),
         "dark": np.array([f["dark"] for f in fps], np.float32),
         "thumb": (np.stack([f["thumb"].reshape(-1) for f in fps]).astype(np.float32) if fps
                   else np.zeros((0, 1024), np.float32)),
         "h": [f["h"] for f in fps], "w": [f["w"] for f in fps]}
    T["matchable"] = (T["dark"] <= DARK) & (T["thumb"].std(1) >= FLAT) & np.array([bool(p) for p in T["pixel"]], bool)
    return T

t0 = time.time()
INS = as_table(fingerprint_all(ins_files, "Instruct"))
BEN = as_table(fingerprint_all(ben_files, "Benchmark"))
FP_MIN = (time.time() - t0) / 60
for T, fname in ((INS, "fingerprints_instruct.csv"), (BEN, "fingerprints_bench.csv")):
    with open(OUT_DIR / fname, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file", "file_md5", "pixel_md5", "dhash", "dark", "height", "width"])
        for k in range(len(T["names"])):
            w.writerow([T["names"][k], T["file"][k], T["pixel"][k], f"{int(T['dhash'][k]):016x}",
                        float(T["dark"][k]), T["h"][k], T["w"][k]])
UNREADABLE = {"instruct": sum(not p for p in INS["pixel"]), "benchmark": sum(not p for p in BEN["pixel"])}
print(f"✓ Fingerprinted {len(INS['names']):,} Instruct and {len(BEN['names']):,} benchmark files in {FP_MIN:.1f} min; "
      f"unreadable: {UNREADABLE}; never near-matched (dark, flat or unreadable): "
      f"{int((~INS['matchable']).sum()):,} and {int((~BEN['matchable']).sum()):,}")

def candidates(A, B, rows=None, cols=None, max_bits=REVIEW_DHASH, chunk=256):
    """All (i, j, dHash distance, thumbnail correlation) with distance <= max_bits, both images matchable."""
    ia = np.arange(len(A["names"])) if rows is None else np.asarray(rows, int)
    jb = np.arange(len(B["names"])) if cols is None else np.asarray(cols, int)
    ia, jb = ia[A["matchable"][ia]], jb[B["matchable"][jb]]
    hi, hj, hd = [], [], []
    for s in range(0, len(ia), chunk):
        blk = ia[s:s + chunk]
        d = bit_distance(A["dhash"][blk][:, None], B["dhash"][jb][None, :])
        ii, jj = np.nonzero(d <= max_bits)
        hi.append(blk[ii])
        hj.append(jb[jj])
        hd.append(d[ii, jj])
    i, j, d = (np.concatenate(h).astype(int) if h else np.zeros(0, int) for h in (hi, hj, hd))
    c = thumb_corr(A["thumb"][i], B["thumb"][j]) if len(i) else np.zeros(0)
    return i, j, d, c

✓ Fingerprint self-checks passed (dHash: recompressed 0, shifted 1, unrelated 32 of 64 bits)
Image files: Instruct 20,988 (same name twice: 0), benchmark 7,278 (same name twice: 0)
  Instruct: 2,000/20,988 (30 files/s; 10.5 min left)
  Instruct: 4,000/20,988 (30 files/s; 9.5 min left)
  Instruct: 6,000/20,988 (30 files/s; 8.3 min left)
  Instruct: 8,000/20,988 (30 files/s; 7.2 min left)
  Instruct: 10,000/20,988 (30 files/s; 6.1 min left)
  Instruct: 12,000/20,988 (31 files/s; 4.8 min left)
  Instruct: 14,000/20,988 (31 files/s; 3.8 min left)
  Instruct: 16,000/20,988 (31 files/s; 2.7 min left)
  Instruct: 18,000/20,988 (31 files/s; 1.6 min left)
  Instruct: 20,000/20,988 (31 files/s; 0.5 min left)
  Instruct: 20,988/20,988 (31 files/s; 0.0 min left)
  Benchmark: 2,000/7,278 (29 files/s; 3.1 min left)
  Benchmark: 4,000/7,278 (30 files/s; 1.8 min left)
  Benchmark: 6,000/7,278 (29 files/s; 0.7 min left)
  Benchmark: 7,278/7,278 (29 files/s; 0.0 min left)
✓ Fingerprinted 20,988 Instruct

## 3. The validation leak
A validation pair leaks, and is **counted**, if either of these holds:
- its before-image has the same pixels as a training pair's before-image. This is the audit's rule: the twins share a before-image, and their after-images are about 88% alike.
- its after-image has the same pixels as a training after-image.

Near-copies of before-images are listed separately as `near copy` or `review`, and never counted.

In [5]:
# ── Cell 5: Validation pairs whose before- or after-image is a training pair's ──
k_of = {n: k for k, n in enumerate(INS["names"])}
for r in POOL:
    r["k_pre"], r["k_post"] = k_of.get(r["pre"]), k_of.get(r["post"])

def pixel_of(k):
    return INS["pixel"][k] if k is not None else ""

# Before-images shared by two or more pool pairs (the audit counts 264 in the pool)
groups = defaultdict(list)
for r in POOL:
    if pixel_of(r["k_pre"]):
        groups[pixel_of(r["k_pre"])].append(r)
twin_groups = {h: rows for h, rows in groups.items() if len(rows) > 1}
pool_rows = [{"before_pixel_md5": h, "pairs": len(rows), "splits": "+".join(sorted(r["split"] for r in rows)),
              "events": "+".join(sorted({r["event"] for r in rows})), "after_images": ";".join(r["post"] for r in rows)}
             for h, rows in twin_groups.items()]

ins_by_pixel = defaultdict(list)
for k, h in enumerate(INS["pixel"]):
    if h:
        ins_by_pixel[h].append(INS["names"][k])
pool_names = {r["pre"] for r in POOL} | {r["post"] for r in POOL}
train_after = defaultdict(list)
for r in TRAIN:
    if pixel_of(r["k_post"]):
        train_after[pixel_of(r["k_post"])].append(r)

def pair_scores(r, t):
    """dHash distance and thumbnail correlation of two pairs' after-images."""
    if r["k_post"] is None or t["k_post"] is None:
        return None, None
    return (int(bit_distance(INS["dhash"][r["k_post"]], INS["dhash"][t["k_post"]])),
            round(float(thumb_corr(INS["thumb"][[r["k_post"]]], INS["thumb"][[t["k_post"]]])[0]), 4))

val_rows, twin_also_in_val, twin_outside_pool = [], 0, 0
for r in VAL:
    h = pixel_of(r["k_pre"])
    same_before = [t for t in twin_groups.get(h, []) if t is not r] if h else []
    in_train = [t for t in same_before if t["split"] == "train"]
    same_after = train_after.get(pixel_of(r["k_post"]), []) if pixel_of(r["k_post"]) else []
    if same_before and not in_train:
        twin_also_in_val += 1
    if h and any(n not in pool_names for n in ins_by_pixel[h] if n != r["pre"]):
        twin_outside_pool += 1
    twins = in_train or same_after
    if twins:
        t = twins[0]
        after_dhash, after_corr = pair_scores(r, t)
        val_rows.append({"val_image": r["post"], "val_before": r["pre"], "event": r["event"],
                         "match": "same before-image" if in_train else "same after-image", "counted": True,
                         "twins_in_training": ";".join(x["post"] for x in twins),
                         "same_file_bytes": INS["file"][r["k_pre"]] == INS["file"][t["k_pre"]] if in_train else None,
                         "before_dhash": 0 if in_train else None, "before_corr": 1.0 if in_train else None,
                         "after_dhash": after_dhash, "after_corr": after_corr, "destroyed_px": r["destroyed_px"],
                         "in_pc_21": r["post"].replace("_post_disaster.png", "") in PC_21})

# Near-copies of before-images (never counted): validation before-images against training before-images
counted = {row["val_image"] for row in val_rows}
val_by_k = {r["k_pre"]: r for r in VAL if r["k_pre"] is not None and r["post"] not in counted}
train_by_k = defaultdict(list)
for r in TRAIN:
    if r["k_pre"] is not None:
        train_by_k[r["k_pre"]].append(r)
best = {}
for i, j, d, c in zip(*candidates(INS, INS, rows=list(val_by_k), cols=list(train_by_k))):
    t = tier(d, c)
    if t and i != j:
        key = (0 if t == "near copy" else 1, int(d), -float(c))
        if i not in best or key < best[i][0]:
            best[i] = (key, t, j, int(d), round(float(c), 4))
for i, (_, t, j, d, c) in best.items():
    r, tw = val_by_k[i], train_by_k[j][0]
    after_dhash, after_corr = pair_scores(r, tw)
    val_rows.append({"val_image": r["post"], "val_before": r["pre"], "event": r["event"], "match": t, "counted": False,
                     "twins_in_training": ";".join(x["post"] for x in train_by_k[j]), "same_file_bytes": False,
                     "before_dhash": d, "before_corr": c, "after_dhash": after_dhash, "after_corr": after_corr,
                     "destroyed_px": r["destroyed_px"],
                     "in_pc_21": r["post"].replace("_post_disaster.png", "") in PC_21})

LEAKED = [r for r in val_rows if r["counted"]]
LEAKED_PX = sum(r["destroyed_px"] for r in LEAKED)
NEAR_ONLY = [r for r in val_rows if r["match"] == "near copy"]
MISSING_PC21 = sorted(set(PC_21) - {r["val_image"].replace("_post_disaster.png", "") for r in LEAKED})
fields = ["val_image", "val_before", "event", "match", "counted", "twins_in_training", "same_file_bytes",
          "before_dhash", "before_corr", "after_dhash", "after_corr", "destroyed_px", "in_pc_21"]
with open(OUT_DIR / "val_train_leaks_hash.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(sorted(val_rows, key=lambda x: (not x["counted"], x["val_image"])))
with open(OUT_DIR / "pool_twin_groups.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["before_pixel_md5", "pairs", "splits", "events", "after_images"])
    w.writeheader()
    w.writerows(pool_rows)
after_corrs = [r["after_corr"] for r in LEAKED if r["after_corr"] is not None]
print(f"✓ Leaked validation pairs (counted): {len(LEAKED)} holding {LEAKED_PX:,} destroyed pixels "
      f"({LEAKED_PX / EXPECTED['val_destroyed_px']:.1%}); near-copies only: {len(NEAR_ONLY)}; "
      f"review: {sum(r['match'] == 'review' for r in val_rows)}")
print(f"  The PC check's 21 among them: {21 - len(MISSING_PC21)} of 21{(' — missing ' + ', '.join(MISSING_PC21)) if MISSING_PC21 else ''}")
print(f"  Twin only in validation (not a leak): {twin_also_in_val}; twin outside the pool: {twin_outside_pool}")
print(f"  Shared before-images in the pool: {len(twin_groups)} groups covering {sum(len(v) for v in twin_groups.values())} pairs "
      f"(audit: {AUDIT['shared_before_images_in_pool']})")
if after_corrs:
    print(f"  Leaked twins' after-images: median correlation {float(np.median(after_corrs)):.2f} (audit: 0.88)")
if SMOKE_TEST:
    assert KNOWN_TWIN[0] in {r["val_image"] for r in LEAKED}, "the known twin wasn't found: check the fingerprints"
    print(f"✓ Smoke test: the known twin {KNOWN_TWIN[0]} ↔ {KNOWN_TWIN[1]} is found")

✓ Leaked validation pairs (counted): 40 holding 155,812 destroyed pixels (8.6%); near-copies only: 0; review: 3
  The PC check's 21 among them: 21 of 21
  Twin only in validation (not a leak): 4; twin outside the pool: 1
  Shared before-images in the pool: 250 groups covering 500 pairs (audit: 264)
  Leaked twins' after-images: median correlation 0.97 (audit: 0.88)


## 4. The benchmark's copies of Instruct images
Every benchmark image is compared with every Instruct image, and its best match falls in one of three tiers:
- **same pixels;**
- **near copy:** passes both thresholds;
- **review:** a looser match, pictured only, never counted.

Each question of the six scored tasks is then flagged in one of two ways:
- **after-image copied:** the model has seen that picture;
- **before-image copied (same place):** only the before-image is a copy, so it's the same place with a new after-image.

The question IDs are the ones H1 V2, L0 and H2 use (`<subset>_<index>`), so your PC can re-score without the flagged questions.

In [6]:
# ── Cell 6: Benchmark images that copy Instruct images, and the questions they touch ──
with open(BENCH_ROOT / "benchmark_release.json", encoding="utf-8") as f:
    bench = json.load(f)

RANK = {"same pixels": 0, "near copy": 1, "review": 2}
match = {}                                       # benchmark file -> its best match in Instruct

def keep_best(name, row):
    old = match.get(name)
    if old is None or (RANK[row["tier"]], row["dhash"], -row["corr"]) < (RANK[old["tier"]], old["dhash"], -old["corr"]):
        match[name] = row

for b, h in enumerate(BEN["pixel"]):
    if h and h in ins_by_pixel:
        keep_best(BEN["names"][b], {"tier": "same pixels", "instruct_file": ins_by_pixel[h][0], "dhash": 0, "corr": 1.0,
                                    "identical_in_instruct": len(ins_by_pixel[h])})
for i, j, d, c in zip(*candidates(BEN, INS)):
    t = tier(d, c)
    if t:
        keep_best(BEN["names"][i], {"tier": t, "instruct_file": INS["names"][j], "dhash": int(d), "corr": round(float(c), 4),
                                    "identical_in_instruct": 0})

# Which benchmark files are before-images and which are after-images
roles = defaultdict(set)
for e in bench:
    if e.get("pre_image_path"):
        roles[name_of(e["pre_image_path"])].add("before")
    for k in ("post_image_path", "image_path"):
        if e.get(k):
            roles[name_of(e[k])].add("after")
copy_rows = [{"bench_file": n, "role": "+".join(sorted(roles.get(n, {"unused"}))), "event": event_and_source(n)[0], **m}
             for n, m in sorted(match.items(), key=lambda t: (RANK[t[1]["tier"]], t[0]))]
with open(OUT_DIR / "bench_copies.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["bench_file", "role", "event", "tier", "instruct_file", "dhash", "corr",
                                      "identical_in_instruct"])
    w.writeheader()
    w.writerows(copy_rows)

# The six scored tasks, with the question IDs H1 V2 and L0 use (as in l0-step2-full-size-rescore.ipynb)
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition", 2),
         ("BBR", "bearing_body", "Disaster Bearing Bodies Recognition", 2),
         ("BDC", "building_damage_counting", "Building Damage Counting", 2),
         ("DRE", "road_damage_counting", "Road Damage Counting", 2),
         ("DSR", "landuse", "Disaster Scene Recognition", 1),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning", 1)]

def sensor(e):
    """H1 V2's rule for the after-image's sensor: most entries have post_image_type, ORR entries image_type."""
    return e.get("post_image_type") or e.get("image_type") or "Optical"

COPIED = {"same pixels", "near copy"}
q_rows, per_task = [], []
for code, subset, task, _ in TASKS:
    entries = [e for e in bench if e.get("task") == task]
    n = {"task": code, "questions": len(entries), "optical": 0, "after_copied": 0, "after_copied_optical": 0,
         "before_only_copied": 0, "before_only_copied_optical": 0, "review_only": 0}
    for i, e in enumerate(entries):
        optical = sensor(e) == "Optical"
        n["optical"] += optical
        after = [name_of(e[k]) for k in ("post_image_path", "image_path") if e.get(k)]
        before = [name_of(e["pre_image_path"])] if e.get("pre_image_path") else []
        a = [match[x] for x in after if x in match and match[x]["tier"] in COPIED]
        b = [match[x] for x in before if x in match and match[x]["tier"] in COPIED]
        rv = [match[x] for x in after + before if x in match and match[x]["tier"] == "review"]
        if a:
            flag, key = "after-image copied", "after_copied"
        elif b:
            flag, key = "before-image copied (same place)", "before_only_copied"
        elif rv:
            flag, key = "review", "review_only"
        else:
            continue
        n[key] += 1
        if optical and key != "review_only":
            n[key + "_optical"] += 1
        m = (a or b or rv)[0]
        q_rows.append({"task": code, "id": f"{subset}_{i}", "sensor": sensor(e), "after_image": ";".join(after),
                       "before_image": ";".join(before), "flag": flag, "tier": m["tier"],
                       "instruct_file": m["instruct_file"], "dhash": m["dhash"], "corr": m["corr"]})
    per_task.append(n)
with open(OUT_DIR / "bench_questions_flagged.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["task", "id", "sensor", "after_image", "before_image", "flag", "tier",
                                      "instruct_file", "dhash", "corr"])
    w.writeheader()
    w.writerows(q_rows)

OPT_TOTAL = sum(t["optical"] for t in per_task)
OPT_AFTER = sum(t["after_copied_optical"] for t in per_task)
SHARE = OPT_AFTER / max(OPT_TOTAL, 1)
tiers = {t: sum(m["tier"] == t for m in match.values()) for t in RANK}
print(f"✓ Benchmark files matched to Instruct: {tiers} (of {len(BEN['names']):,} benchmark files)")
for t in per_task:
    print(f"  {t['task']}: {t['questions']:,} questions ({t['optical']:,} optical); after-image copied "
          f"{t['after_copied']} ({t['after_copied_optical']} optical); before-image only {t['before_only_copied']}; "
          f"review {t['review_only']}")
print(f"  Optical questions with a copied after-image: {OPT_AFTER:,} of {OPT_TOTAL:,} ({SHARE:.2%}); "
      f"O2 needs a clean re-score: {'yes' if SHARE > 0.01 else 'no'} (threshold 1%, fixed before the run)")

✓ Benchmark files matched to Instruct: {'same pixels': 519, 'near copy': 39, 'review': 48} (of 7,278 benchmark files)
  DTR: 420 questions (353 optical); after-image copied 10 (10 optical); before-image only 6; review 2
  BBR: 2,363 questions (2,006 optical); after-image copied 35 (35 optical); before-image only 24; review 19
  BDC: 4,982 questions (3,583 optical); after-image copied 188 (66 optical); before-image only 44; review 24
  DRE: 2,178 questions (1,868 optical); after-image copied 34 (34 optical); before-image only 23; review 14
  DSR: 2,007 questions (2,007 optical); after-image copied 35 (35 optical); before-image only 24; review 18
  ORR: 1,018 questions (858 optical); after-image copied 6 (6 optical); before-image only 0; review 6
  Optical questions with a copied after-image: 186 of 10,675 (1.74%); O2 needs a clean re-score: yes (threshold 1%, fixed before the run)


In [7]:
# ── Cell 7: Pictures for a check by eye (matches side by side) ──
ins_path = dict(zip(INS["names"], INS["paths"]))
ben_path = dict(zip(BEN["names"], BEN["paths"]))
pre_of = {r["post"]: r["pre"] for r in POOL}

def side_by_side(path_a, path_b, title, out_path, size=384):
    a, b = cv2.imread(path_a), cv2.imread(path_b)
    if a is None or b is None:
        return False
    a, b = (cv2.resize(x, (size, size), interpolation=cv2.INTER_AREA) for x in (a, b))
    canvas = np.full((size + 40, 2 * size + 8, 3), 255, np.uint8)
    canvas[40:, :size], canvas[40:, size + 8:] = a, b
    cv2.putText(canvas, title[:120], (4, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 0, 0), 1, cv2.LINE_AA)
    return bool(cv2.imwrite(str(out_path), canvas))

PICTURES = []
bench_exact = [(n, m) for n, m in match.items() if m["tier"] == "same pixels"][:4]
bench_near = sorted(((n, m) for n, m in match.items() if m["tier"] != "same pixels"), key=lambda t: -t[1]["corr"])[:30]
for n, m in bench_exact + bench_near:
    out = OUT_DIR / "review" / f"bench_{len(PICTURES):02d}_{m['tier'].replace(' ', '-')}.png"
    title = f"{m['tier']} | dHash {m['dhash']} | r {m['corr']:.3f} | benchmark {n} | Instruct {m['instruct_file']}"
    if side_by_side(ben_path[n], ins_path[m["instruct_file"]], title, out):
        PICTURES.append(out.name)
val_pics = [r for r in val_rows if r["counted"]][:4] + [r for r in val_rows if not r["counted"]][:10]
for r in val_pics:
    twin = r["twins_in_training"].split(";")[0]
    a, b = (r["val_before"], pre_of[twin]) if r["match"] != "same after-image" else (r["val_image"], twin)
    out = OUT_DIR / "review" / f"val_{len(PICTURES):02d}_{r['match'].replace(' ', '-')}.png"
    title = f"{r['match']} | validation {r['val_image']} | training {twin}"
    if a in ins_path and b in ins_path and side_by_side(ins_path[a], ins_path[b], title, out):
        PICTURES.append(out.name)
print(f"✓ {len(PICTURES)} pictures in {OUT_DIR / 'review'}: the left image is the benchmark or validation one, "
      "the right one its match in training")

✓ 41 pictures in /kaggle/working/E5-step1/review: the left image is the benchmark or validation one, the right one its match in training


In [8]:
# ── Cell 8: Results against the expectation, and the output files ──
import pandas as pd

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

by_event = pd.DataFrame(copy_rows).groupby(["event", "tier"]).size().unstack(fill_value=0).reset_index() \
    if copy_rows else pd.DataFrame(columns=["event"])
checks = {
    "leaked pairs within 21-40": "n/a (smoke test)" if SMOKE_TEST else ("yes" if 21 <= len(LEAKED) <= 40 else "NO"),
    "PC check's 21 all found": "n/a (smoke test)" if SMOKE_TEST else ("yes" if not MISSING_PC21 else "NO"),
    "O2 needs a clean re-score (over 1% of optical questions with a copied after-image)":
        "n/a (smoke test)" if SMOKE_TEST else ("yes" if SHARE > 0.01 else "no"),
}
summary = {
    "task": "E5 step 1", "smoke_test": SMOKE_TEST, "fingerprint_minutes": round(FP_MIN, 1),
    "files": {"instruct": len(INS["names"]), "benchmark": len(BEN["names"]), "unreadable": UNREADABLE,
              "same_name_twice": {"instruct": ins_dup, "benchmark": ben_dup}},
    "thresholds": {"near_dhash": NEAR_DHASH, "near_corr": NEAR_CORR, "review_dhash": REVIEW_DHASH,
                   "review_corr": REVIEW_CORR, "dark": DARK, "flat_std": FLAT},
    "validation": {"pairs": len(VAL), "destroyed_px": val_destroyed, "leaked_counted": len(LEAKED),
                   "leaked_destroyed_px": LEAKED_PX, "by_match": {m: sum(r["match"] == m for r in val_rows)
                                                                  for m in ("same before-image", "same after-image",
                                                                            "near copy", "review")},
                   "pc_21_missing": MISSING_PC21, "twin_only_in_validation": twin_also_in_val,
                   "twin_outside_pool": twin_outside_pool,
                   "after_image_corr_median": round(float(np.median(after_corrs)), 3) if after_corrs else None},
    "pool": {"shared_before_image_groups": len(twin_groups), "pairs_in_groups": sum(len(v) for v in twin_groups.values())},
    "audit": AUDIT,
    "benchmark": {"files_matched": tiers, "per_task": per_task, "optical_questions": OPT_TOTAL,
                  "optical_after_copied": OPT_AFTER, "share": round(SHARE, 4)},
    "checks": checks, "pictures": len(PICTURES),
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")

lines = [
    "# E5 step 1: the validation leak and the benchmark's copies of training images", "",
    f"{'**SMOKE TEST** (a sample of files): checks the code, not the answer. ' if SMOKE_TEST else ''}"
    f"{len(INS['names']):,} DisasterM3 Instruct files and {len(BEN['names']):,} benchmark files were fingerprinted "
    f"in {FP_MIN:.1f} min.", "",
    "## The validation leak (the 645 pairs every D4 run is scored on)", "",
    f"- **Leaked pairs (counted): {len(LEAKED)}**, holding **{LEAKED_PX:,}** of {val_destroyed:,} destroyed pixels "
    f"({LEAKED_PX / val_destroyed:.1%}). The audit says {AUDIT['leaked_pairs']} and {AUDIT['leaked_destroyed_px']:,}; "
    "the PC check (xBD's original files) found 21 and 80,709.",
    f"  - same before-image as a training pair: {summary['validation']['by_match']['same before-image']}; "
    f"same after-image: {summary['validation']['by_match']['same after-image']}",
    f"- The PC check's 21 among them: {21 - len(MISSING_PC21)} of 21"
    f"{(' (missing: ' + ', '.join(MISSING_PC21) + ')') if MISSING_PC21 else ''}.",
    f"- Near-copies of before-images, not counted: {len(NEAR_ONLY)} near copy, "
    f"{summary['validation']['by_match']['review']} review (pictures in `review/`).",
    f"- Twins that are both in validation (no leak): {twin_also_in_val}; twins outside the pool: {twin_outside_pool}.",
    f"- Shared before-images in the pool: {len(twin_groups)} groups (the audit: {AUDIT['shared_before_images_in_pool']}).",
    f"- Leaked twins' after-images correlate at a median of "
    f"{summary['validation']['after_image_corr_median']} (the audit: 0.88).", "",
    "## The benchmark's copies of Instruct images", "",
    f"- Benchmark files with a match in Instruct: {tiers['same pixels']} same pixels, {tiers['near copy']} near copy, "
    f"{tiers['review']} review only.",
    f"- **Optical questions of the six tasks whose after-image is a copy: {OPT_AFTER:,} of {OPT_TOTAL:,} ({SHARE:.2%}).**",
    "", as_markdown(pd.DataFrame(per_task)), "",
    "By event:", "", as_markdown(by_event), "",
    "## Against the expectation (fixed before the run, 26 Oct)", "",
    *[f"- {k}: **{v}**" for k, v in checks.items()], "",
    "## How it was checked, and the alternatives", "",
    "- **Same pixels** (the counted rule): an MD5 of the decoded pixels, so a copy saved again with other file bytes "
    "still matches. The file MD5 is kept too (`same_file_bytes`).",
    f"- **Near copy** (listed, not counted): dHash (64 bits from a 9 × 8 grey thumbnail) within {NEAR_DHASH} bits "
    f"AND 32 × 32 thumbnails correlating at {NEAR_CORR} or more. **Review**: within {REVIEW_DHASH} bits and "
    f"{REVIEW_CORR} or more. Images over half black or nearly flat are never near-matched, since dHash collides on them.",
    "- **Not used:**",
    "  - matching file names, which misses copies under another name; that is how D5's check missed this leak;",
    "  - building masks: twins' masks differ (IoU 0.72), so they can't find copies;",
    "  - local features (SIFT, ORB) or CNN embeddings, which survive bigger crops and shifts. Add them only if the "
    "pictures show copies these thresholds miss.",
    "- **The split** is rebuilt with R11's verbatim cells and checked: 6,443 pairs, 645 validation images "
    "(560 xBD), 1,808,111 destroyed pixels.", "",
    "## Files", "",
    "- `val_train_leaks_hash.csv`: every validation pair with a match, and whether it's counted",
    "- `pool_twin_groups.csv`: before-images shared by two or more pool pairs",
    "- `bench_copies.csv`: every benchmark file with a match in Instruct",
    "- `bench_questions_flagged.csv`: the six tasks' questions that use a copied image (IDs as in H1 V2 and L0)",
    "- `fingerprints_instruct.csv`, `fingerprints_bench.csv`: every file's fingerprints, for later checks without a rerun",
    "- `review/`: pictures, the benchmark or validation image on the left and its match in training on the right",
    "- `summary.json`", "",
    "## Next, on the PC", "",
    "- Re-score R16c (steps 2 and 5) and, with E5 step 2's per-image counts, R11, R11s7, R9 and R1 without the "
    "counted pairs.",
    "- If the benchmark share is over 1%: re-score the 7B's six tasks (L0, H1 V2) and H2 without the flagged questions.",
]
(OUT_DIR / "README.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:16]))
print(f"\n✓ Wrote {OUT_DIR}: README.md, summary.json, val_train_leaks_hash.csv, pool_twin_groups.csv, bench_copies.csv, "
      "bench_questions_flagged.csv, fingerprints_instruct.csv, fingerprints_bench.csv, review/")

# E5 step 1: the validation leak and the benchmark's copies of training images

20,988 DisasterM3 Instruct files and 7,278 benchmark files were fingerprinted in 15.4 min.

## The validation leak (the 645 pairs every D4 run is scored on)

- **Leaked pairs (counted): 40**, holding **155,812** of 1,808,111 destroyed pixels (8.6%). The audit says 40 and 155,812; the PC check (xBD's original files) found 21 and 80,709.
  - same before-image as a training pair: 40; same after-image: 0
- The PC check's 21 among them: 21 of 21.
- Near-copies of before-images, not counted: 0 near copy, 3 review (pictures in `review/`).
- Twins that are both in validation (no leak): 4; twins outside the pool: 1.
- Shared before-images in the pool: 250 groups (the audit: 264).
- Leaked twins' after-images correlate at a median of 0.966 (the audit: 0.88).

## The benchmark's copies of Instruct images


✓ Wrote /kaggle/working/E5-step1: README.md, summary.json, val_train_leaks_hash.csv, pool_twin_groups.csv, bench_